# 09 - Rally Analysis: Event Spotting (PES) + per-shot classification

Upload a **full / multi-shot rally clip** (e.g. 10-20 s). The notebook:
1. Builds a **motion-energy signal** over the video.
2. **Detects hits** as peaks in that signal (Precise Event Spotting).
3. **Segments** the rally into shots and **classifies each** with model23.
4. Builds a **rally timeline** + shot-mix + next-shot + coaching, and plots the hits.

Note: this is event spotting via **motion** (robust, runs today). True shuttle
tracking with **TrackNetV2** is a drop-in upgrade for the signal in cell 4.

Keep clips short-ish (memory). Run cells 1 -> 5; set VIDEO in cell 5 to your clip.

In [9]:
import torch, os
assert torch.cuda.is_available(), 'No GPU active. Runtime > Change runtime type > L4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print('Drive ready:', os.path.isdir('/content/drive/MyDrive'))
!pip install -q opencv-python-headless scipy

GPU: NVIDIA L4
Drive ready: True


In [10]:
CLASSES = [
    (0, 'Short Serve', '00_Short Serve'),
    (1, 'Lift',        '02_Lift'),
    (2, 'Clear',       '12_Clear'),
    (3, 'Drop Shot',   '05_Drop Shot'),
    (4, 'Smash',       '14_Smash'),
]
NUM_CLASSES = len(CLASSES)
ID_TO_NAME = {c[0]: c[1] for c in CLASSES}
NAME_TO_ID = {c[1]: c[0] for c in CLASSES}
PAD_TOKEN = NUM_CLASSES
SEQ_LEN = 4
NUM_FRAMES = 16
IMG = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DRIVE = '/content/drive/MyDrive/BadmintonFYP'
MODELS = DRIVE + '/model23'
VIDEO_CKPT = MODELS + '/video_model.pth'
SEQUENCE_CKPT = MODELS + '/sequence_model.pth'

COACH = {
    'Smash': 'Attack played - opponent likely forced into ',
    'Drop Shot': 'Net pressure applied - expect ',
    'Short Serve': 'Rally started - anticipate ',
    'Lift': 'Lifted to the back - prepare for ',
    'Clear': 'Cleared deep - next likely ',
}

MOTION_MAX_DIM = 320
SMOOTH_WIN = 5
MIN_GAP_S = 0.5
SENSITIVITY = 0.45
WINDOW_S = 0.6

import os
print('Models:', MODELS, '| video exists:', os.path.exists(VIDEO_CKPT))

Models: /content/drive/MyDrive/BadmintonFYP/model23 | video exists: True


In [11]:
import cv2, numpy as np, torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TemporalPoolingResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        base = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b*t, c, h, w)
        x = self.features(x).view(b, t, 512).mean(dim=1)
        return self.fc(x)

class ShotPredictorGRU(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed=32, hidden=128, layers=2, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(num_classes+1, embed, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(embed, hidden, layers, batch_first=True, dropout=drop if layers>1 else 0)
        self.fc = nn.Sequential(nn.Linear(hidden,hidden), nn.ReLU(), nn.Dropout(drop), nn.Linear(hidden,num_classes))
    def forward(self, x):
        x = self.emb(x); o,_ = self.gru(x); return self.fc(o[:,-1,:])

_norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
def letterbox(img, size=IMG):
    h, w = img.shape[:2]; s = min(size/h, size/w); nh, nw = int(h*s), int(w*s)
    r = cv2.resize(img, (nw, nh)); c = np.zeros((size,size,3),np.uint8); y, x = (size-nh)//2, (size-nw)//2; c[y:y+nh, x:x+nw] = r; return c

def _load(model, path):
    ck = torch.load(path, map_location=DEVICE)
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    model.load_state_dict(state); model.eval(); return model

video_model = _load(TemporalPoolingResNet().to(DEVICE), VIDEO_CKPT)
print('loaded video model')
seq_model = None
if os.path.exists(SEQUENCE_CKPT):
    seq_model = _load(ShotPredictorGRU().to(DEVICE), SEQUENCE_CKPT)
    print('loaded sequence model')
else:
    print('sequence model missing - next-shot disabled')

loaded video model
loaded sequence model


In [12]:
def read_rally(path, max_dim=MOTION_MAX_DIM):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        h, w = fr.shape[:2]
        s = max_dim / max(h, w)
        if s < 1.0:
            fr = cv2.resize(fr, (int(w*s), int(h*s)))
        frames.append(fr)
    cap.release()
    return frames, float(fps)

def motion_signal(frames):
    g = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY).astype(np.float32) for f in frames]
    m = np.zeros(len(frames))
    for i in range(1, len(frames)):
        m[i] = np.abs(g[i] - g[i-1]).mean()
    return m

def smooth(x, k=SMOOTH_WIN):
    if k <= 1: return x
    return np.convolve(x, np.ones(k)/k, mode='same')

def detect_hits(m, fps):
    sm = smooth(m, SMOOTH_WIN)
    base = np.percentile(sm, 50); hi = np.percentile(sm, 95)
    prom = max(1e-6, SENSITIVITY * (hi - base))
    dist = max(1, int(MIN_GAP_S * fps))
    peaks, _ = find_peaks(sm, distance=dist, prominence=prom)
    return peaks, sm

def window_frames(frames, center, fps):
    half = int(WINDOW_S * fps)
    a = max(0, center - half); b = min(len(frames)-1, center + half)
    if b <= a: b = min(len(frames)-1, a+1)
    idxs = np.linspace(a, b, NUM_FRAMES).astype(int)
    return [frames[i] for i in idxs]

def classify(win):
    x = torch.stack([_norm(torch.from_numpy(letterbox(f).astype(np.float32)/255.0).permute(2,0,1)) for f in win]).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(video_model(x), dim=1)[0].cpu()
    i = int(probs.argmax())
    return i, float(probs[i])

def predict_next(ids):
    if seq_model is None: return None
    h = list(ids)[-SEQ_LEN:]
    if len(h) < SEQ_LEN: h = [PAD_TOKEN]*(SEQ_LEN-len(h)) + h
    with torch.no_grad():
        p = torch.softmax(seq_model(torch.tensor([h], dtype=torch.long, device=DEVICE)), dim=1)[0].cpu()
    v, idx = p.topk(3)
    return [(ID_TO_NAME[int(j)], float(val)) for val, j in zip(v, idx)]

def analyze_rally(path):
    frames, fps = read_rally(path)
    m = motion_signal(frames)
    peaks, sm = detect_hits(m, fps)
    timeline, ids = [], []
    for pk in peaks:
        i, conf = classify(window_frames(frames, pk, fps))
        ids.append(i)
        timeline.append({'t': pk / fps, 'frame': int(pk), 'shot': ID_TO_NAME[i], 'conf': conf})
    return frames, fps, m, sm, peaks, timeline, ids

print('rally pipeline ready')

rally pipeline ready


In [ ]:

COACH_LINE = {
    'Short Serve': 'Serve is in - get back to base and watch the net.',
    'Lift':        'You are on the defensive now - get low and ready for the smash.',
    'Clear':       'Good deep clear - recover to centre and stay balanced.',
    'Drop Shot':   'Nice drop - follow it in and pressure the net.',
    'Smash':       'Strong smash - move in and look to finish at the net.',
}

NEXT_HINT = {
    'Short Serve': 'watch for a net reply',
    'Lift':        'be ready to defend the smash',
    'Clear':       'expect it deep to the back court',
    'Drop Shot':   'cover the net',
    'Smash':       'brace and prepare to defend',
}

def coach_advice(timeline, ids, fps, total_frames, nxt):
    cur = timeline[-1]['shot']
    line = COACH_LINE.get(cur, 'Shot played.')
    if nxt:
        nshot = nxt[0][0]
        line += ' Likely next: %s - %s.' % (nshot, NEXT_HINT.get(nshot, 'stay ready'))
    return line

print('coaching engine ready')

In [ ]:
from collections import Counter
import glob

VIDEO = DRIVE + '/0609.mp4'

if not os.path.isfile(VIDEO):
    cands = []
    for d in ['/content', DRIVE, '/content/drive/MyDrive']:
        for ext in ('mp4', 'mov', 'avi', 'mkv', 'MP4', 'MOV', 'AVI', 'MKV'):
            cands += glob.glob(d + '/*.' + ext)
    cands = sorted(set(c for c in cands if os.path.isfile(c)))
    assert cands, 'VIDEO not found and no other video in /content, BadmintonFYP, or MyDrive.'
    VIDEO = max(cands, key=os.path.getmtime)
print('Using:', VIDEO)

frames, fps, m, sm, peaks, timeline, ids = analyze_rally(VIDEO)
print('Duration %.1fs | %d frames | %.0f fps | shots detected: %d' % (len(frames)/fps, len(frames), fps, len(peaks)))

if len(peaks) == 0:
    print('No hits detected. Lower SENSITIVITY (cell 2) and re-run, or check the clip.')
else:
    print()
    print('RALLY TIMELINE')
    for j, ev in enumerate(timeline, 1):
        print('  %2d. %5.1fs  %-12s (%.0f%%)' % (j, ev['t'], ev['shot'], 100*ev['conf']))
    print()
    print('Shot mix:', dict(Counter(ev['shot'] for ev in timeline)))
    print('Tempo: %.2f s/shot' % ((len(frames)/fps) / max(len(peaks), 1)))
    nxt = predict_next(ids)
    if nxt:
        print('Predicted next shot:', [(l, round(p, 2)) for l, p in nxt])
    print()
    print('COACH:', coach_advice(timeline, ids, fps, len(frames), nxt))

    t = np.arange(len(sm)) / fps
    plt.figure(figsize=(11, 4))
    plt.plot(t, sm, color='C0', label='motion energy (smoothed)')
    plt.scatter(np.array(peaks) / fps, sm[peaks], color='C3', zorder=5, label='detected hits')
    for pk in peaks:
        plt.axvline(pk / fps, color='C3', alpha=0.25)
    plt.xlabel('time (s)'); plt.ylabel('motion'); plt.title('Precise Event Spotting - detected hits')
    plt.legend(); plt.tight_layout()
    plt.savefig(MODELS + '/rally_pes.png', dpi=150); plt.show()
    print('Saved plot to', MODELS + '/rally_pes.png')